In [1]:
from langchain_community.document_loaders import TextLoader , PyPDFLoader
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints.embeddings import NVIDIAEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatMessagePromptTemplate
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
model = ChatNVIDIA(model="mistralai/mistral-nemotron")

In [3]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

print(ChatNVIDIA.get_available_models())

[Model(id='qwen/qwen2.5-coder-7b-instruct', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=False, supports_structured_output=False, supports_thinking=False, thinking_prefix=None, no_thinking_prefix=None, thinking_param_enable=None, thinking_param_disable=None, base_model=None), Model(id='ibm/granite-3.3-8b-instruct', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=True, supports_structured_output=False, supports_thinking=True, thinking_prefix=None, no_thinking_prefix=None, thinking_param_enable={'chat_template_kwargs': {'enable_thinking': True}}, thinking_param_disable={'chat_template_kwargs': {'enable_thinking': False}}, base_model=None), Model(id='thudm/chatglm3-6b', model_type='chat', client='ChatNVIDIA', endpoint=None, aliases=None, supports_tools=False, supports_structured_output=False, supports_thinking=False, thinking_prefix=None, no_thinking_prefix=None, thinking_param_enable=None, thinking_param_disable=N

In [4]:
template = ChatPromptTemplate.from_messages([
    ("system", "You summarize the given text."),
    ("human", "{data}")
])

1

In [5]:
data = PyPDFLoader("book.pdf")
docs = data.load()

2

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)



In [7]:
prompts = template.format_messages(data=docs[22].page_content)

3. Splitting the documents 

In [8]:
chunks = splitter.split_documents(docs)
chunks 

[Document(metadata={'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-18T12:52:36+08:00', 'author': 'Christopher M. Bishop', 'moddate': '2008-02-08T16:41:33+01:00', 'title': 'Pattern Recognition and Machine Learning', 'source': 'book.pdf', 'total_pages': 758, 'page': 1, 'page_label': 'ii'}, page_content='Information Science and Statistics\nSeries Editors:\nM. Jordan\nJ. Kleinberg\nB. Scho¨lkopf'),
 Document(metadata={'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-18T12:52:36+08:00', 'author': 'Christopher M. Bishop', 'moddate': '2008-02-08T16:41:33+01:00', 'title': 'Pattern Recognition and Machine Learning', 'source': 'book.pdf', 'total_pages': 758, 'page': 2, 'page_label': 'iii'}, page_content='Information Science and Statistics \nAkaike and Kitagawa: The Practice of Time Series Analysis. \nBishop:  Pattern Recognition and Machine Learning. \nCowell, Dawid, Lauritzen, and Spi

4. Creating Embedding models 

In [9]:
# Filter out empty or whitespace only pages
docs = [doc for doc in docs if doc.page_content.strip()]

print(f"Loaded {len(docs)} non-empty pages")

Loaded 748 non-empty pages


In [10]:
embedding_model = NVIDIAEmbeddings(model = "baai/bge-m3")

In [11]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding = embedding_model,
    persist_directory = "CHROMA-DB"
)

Creating Retrieval

In [12]:
retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k" : 4,
        "fetch_k" : 10,
        "lambda_mult" : 0.5
    }
)

In [13]:
llm = ChatNVIDIA(model="mistralai/mistral-nemotron")

Creating Prompt TEmp

In [14]:
prompts = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful AI assistant.
Use ONLY the provided context to answer the question.
If the answer is not present in the context,
say: "I could not find the answer in the document."
"""
        ),
        (
            "human",
            """Context:
{context}

Question:
{question}
"""
        )
    ]
)

print("Rag System Created")
print("Press ) to Exit")

Rag System Created
Press ) to Exit


In [15]:
while True :
    query = input("You: ")
    if query == "0":
        break
    docs = retriever.invoke(query)

    context = "\n\n".join(
        [docs.page_content for docs in docs]
    )

    final_prompt = prompts.invoke({
        "context":context,
        "question":query
    })

    responce = llm.invoke(final_prompt)

    print(f"Ai :{responce.content}")

Ai :Probability theory provides a consistent framework for the quantification and manipulation of uncertainty and forms one of the central foundations for pattern recognition.
